In [ ]:
# --- Ensure working directory is project root (contains 'parameter/') ---
import os
if not os.path.isdir('parameter') and os.path.isdir('../parameter'):
    os.chdir('..')


In [ ]:
# --- Pick per-case data via CASE_ID env var ---
import os
CASE_ID = os.environ.get('CASE_ID')
if CASE_ID is None:
    raise RuntimeError("CASE_ID env var must be set (e.g. 'case0_N5').")
print(f'Running case: {CASE_ID}')


In [ ]:
import numpy as np
from numpy.linalg import norm
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pickle
from matplotlib.patches import Polygon,Circle


 # Nominal ACS Flocking (pure flocking control only, no CBF)
 # File intended to be used as a baseline for comparison

In [ ]:

# ==============================================================================
# 1. System dynamics definition (multi-agent)
# ==============================================================================

def multi_agent_const_speed_5d_dynamics(t, y, u_all_flat, n_agents, state_dim, V_const):
    """
    Dynamics of N_AGENTS 5D constant-speed models.
    State: [px, py, vx, vy, theta], control: [omega]
    """
    current_states = y.reshape((state_dim, n_agents), order='F')
    control_inputs = u_all_flat.reshape((1, n_agents), order='F')

    vx = current_states[2, :]
    vy = current_states[3, :]
    theta = current_states[4, :]
    omega = control_inputs[0, :]

    d_state = np.zeros_like(current_states)
    d_state[0, :] = vx
    d_state[1, :] = vy
    d_state[2, :] = -V_const * np.sin(theta) * omega
    d_state[3, :] =  V_const * np.cos(theta) * omega
    d_state[4, :] = omega

    return d_state.flatten('F')


In [ ]:

def relative_state(STATE_DIM, N_AGENTS, current_states_matrix):
    """
    Vectorized version of the function that computes relative states between all agent pairs.
    """
    p = current_states_matrix[0:2, :]
    v = current_states_matrix[2:4, :]
    seta = current_states_matrix[4, :]

    relative_p_tensor = p[:, :, np.newaxis] - p[:, np.newaxis, :]
    relative_v_tensor = v[:, :, np.newaxis] - v[:, np.newaxis, :]

    norm_relative_p = np.linalg.norm(relative_p_tensor, axis=0)
    norm_relative_v = np.linalg.norm(relative_v_tensor, axis=0)
    dot_relative_pv = np.sum(relative_p_tensor * relative_v_tensor, axis=0)
    relative_seta = np.sin(seta[np.newaxis, :] - seta[:, np.newaxis])

    vector_seta_all = np.vstack([-np.sin(seta), np.cos(seta)])
    dot_relative_p_seta = np.sum(vector_seta_all[:, :, np.newaxis] * (-relative_p_tensor), axis=0)

    return norm_relative_p, norm_relative_v, dot_relative_pv, relative_seta, dot_relative_p_seta

In [ ]:

def ACS_flocking(u_limit, beta, lamda, k1, k2, desired_distance, N_AGENTS, V_CONST, pij_norm, vij_norm, pv_norm, seta_ij, p_dot_seta):
    """
    Vectorized version of the ACS_flocking algorithm.
    """
    align_weight_matrix = (1 + (pij_norm)**2)**(-beta)
    u_align_all = (lamda / N_AGENTS) * np.sum(align_weight_matrix * seta_ij, axis=1)

    pij_norm_safe = pij_norm.copy()
    np.fill_diagonal(pij_norm_safe, 1.0)

    term_A_matrix = (k1 / (2 * pij_norm_safe**2)) * pv_norm
    term_B_matrix = (k2 * (pij_norm_safe - desired_distance) / (2 * pij_norm_safe)) * p_dot_seta
    combined_matrix = term_A_matrix + term_B_matrix
    np.fill_diagonal(combined_matrix, 0)

    u_inter_all = (1 / (N_AGENTS * V_CONST)) * np.sum(combined_matrix, axis=1)

    flocking_control = u_align_all + u_inter_all
    flocking_control = np.clip(flocking_control, -u_limit, u_limit)
    flocking_control = np.trunc(flocking_control * 1000) / 1000

    return flocking_control.reshape(-1, 1)

In [ ]:
def minimum_inter_distance_function(pij_norm, N_AGENTS):
    pij_with_inf_diag = pij_norm.copy()
    np.fill_diagonal(pij_with_inf_diag, np.inf)
    return np.min(pij_with_inf_diag)

In [ ]:
def maximum_inter_distance_function(pij_norm, N_AGENTS):
    pij_with_inf_diag = pij_norm.copy()
    np.fill_diagonal(pij_with_inf_diag, -1*np.inf)
    return np.max(pij_with_inf_diag)

In [ ]:
def calculate_std_dev(states_matrix, n_agents):
    """
    Computes the spatial standard deviation of a given state matrix (position or velocity).
    """
    if n_agents == 0:
        return 0.0
    mean_vec = np.mean(states_matrix, axis=1, keepdims=True)
    centered_matrix = states_matrix - mean_vec
    sum_of_squared_distances = np.sum(centered_matrix**2)
    std_dev = np.sqrt(sum_of_squared_distances / n_agents)
    return std_dev

In [ ]:

# ==============================================================================
# 2. Simulation loop (Nominal Flocking - no CBF)
# ==============================================================================
def run_multi_agent_simulation(N_AGENTS, V_CONST, critical_distance, initial_states_matrix):
    # --- Simulation parameters ---
    feasible = 1

    position_std_dev = calculate_std_dev(initial_states_matrix[0:2, :], N_AGENTS)
    velocity_std_dev = calculate_std_dev(initial_states_matrix[2:4, :], N_AGENTS)

    T_FINAL = 600
    DT_CONTROL = 0.05

    # -- Vehicle parameters --
    u_limit = 0.35  # rad

    # -- ACS parameters
    beta = np.load('parameter/beta.npy')
    lamda = np.load('parameter/lamda.npy')
    k1 = np.load('parameter/k1.npy')
    k2 = np.load('parameter/k2.npy')
    desired_distance = np.load(f'parameter/{CASE_ID}/desired_distance.npy')

    y0 = initial_states_matrix.flatten('F')

    # --- Simulation variables ---
    times = np.arange(0, T_FINAL, DT_CONTROL)
    history = [initial_states_matrix]
    control_history = []
    minimum_distance_history = []
    maximum_distance_history = []
    collision_time = None
    current_y = y0

    print("5D Simulation starting (Nominal Flocking - No CBF)...")

    for t in times[:-1]:

        # Print progress (every 100 s)
        if t > 0 and t % 50 < DT_CONTROL:
            print(f"  Progress: t={t:.0f}/{T_FINAL}s ({t/T_FINAL*100:.1f}%)")

        current_states_matrix = current_y.reshape((STATE_DIM, N_AGENTS), order='F')
        current_states_matrix[4, :] = (current_states_matrix[4, :] + np.pi) % (2 * np.pi) - np.pi

        # relative state information build
        pij_norm, vij_norm, pv_norm, seta_ij, p_dot_seta = relative_state(STATE_DIM, N_AGENTS, current_states_matrix)

        # Calculate minimum distance
        minimum_inter_distance = minimum_inter_distance_function(pij_norm, N_AGENTS)
        minimum_distance_history.append(minimum_inter_distance)

        # Collision detection (no abort — NOM is baseline; record first collision time only)
        if collision_time is None and minimum_inter_distance < critical_distance:
            collision_time = float(t)
            print(f"COLLISION at t={t:.3f}: min_dist={minimum_inter_distance:.4f} < critical_distance={float(critical_distance):.4f}")

        # Calculate maximum distance
        maximum_inter_distance = maximum_inter_distance_function(pij_norm, N_AGENTS)
        maximum_distance_history.append(maximum_inter_distance)

        # Nominal flocking control (applied directly, no CBF)
        u_nominal = ACS_flocking(u_limit, beta, lamda, k1, k2, desired_distance,
                                N_AGENTS, V_CONST, pij_norm, vij_norm, pv_norm, seta_ij, p_dot_seta)

        optimal_u = u_nominal  # Use nominal directly, no CBF filter

        # Save control input
        control_history.append(optimal_u.copy())

        # --- Numerical integration (over one control period) ---
        sol = solve_ivp(
            fun=multi_agent_const_speed_5d_dynamics,
            t_span=[t, t + DT_CONTROL],
            y0=current_y,
            args=(optimal_u, N_AGENTS, STATE_DIM, V_CONST)
        )

        current_y = sol.y[:, -1]
        history.append(current_y.reshape((STATE_DIM, N_AGENTS), order='F'))

    print("Simulation finished.")
    print("feasible:", feasible)

    # Compute std dev of the final state
    final_states_matrix = history[-1]
    final_position_std_dev = calculate_std_dev(final_states_matrix[0:2, :], N_AGENTS)
    final_velocity_std_dev = calculate_std_dev(final_states_matrix[2:4, :], N_AGENTS)

    return (times, np.array(history), np.array(control_history),
            np.array(minimum_distance_history), np.array(maximum_distance_history),
            feasible, position_std_dev, velocity_std_dev,
            final_position_std_dev, final_velocity_std_dev,
            collision_time)


In [ ]:
# --- Initial state setup (5-dimensional) ---

N_AGENTS = np.load(f'parameter/{CASE_ID}/N_AGENTS.npy')
V_CONST = np.load('parameter/V_CONST.npy')
critical_distance = np.load(f'parameter/{CASE_ID}/critical_distance.npy')
STATE_DIM = np.load('parameter/STATE_DIM.npy')
initial_test_case = np.load(f'initial_conditions/{CASE_ID}/initial.npy')
test_case_num = int(initial_test_case.shape[-1])   # derived from initial.npy shape

In [ ]:
nominal_total_history = []
nominal_total_control_history = []
nominal_total_minimum_distance_history = []
nominal_total_maximum_distance_history = []
nominal_feasibility_list = []
nominal_failure_reason_list = []
nominal_failure_time_list = []
nominal_initial_position_std_dev_list = []
nominal_initial_velocity_std_dev_list = []
nominal_final_position_std_dev_list = []
nominal_final_velocity_std_dev_list = []

In [ ]:
# Run simulation
for test_case in range(test_case_num):
    print(f"Running simulation for test case {test_case}...")
    initial_states_matrix = initial_test_case[:, :, test_case]
    (times, history, heading_angel_rate, smallist_distance, largiest_distance,
     feasiblity, initial_position_std_dev, initial_velocity_std_dev,
     final_position_std_dev, final_velocity_std_dev, collision_time) = run_multi_agent_simulation(
        N_AGENTS, V_CONST, critical_distance, initial_states_matrix
    )

    # Nominal flocking is always feasible (no CBF means no infeasibility)
    nominal_total_history.append(history)
    nominal_total_control_history.append(heading_angel_rate)
    nominal_total_minimum_distance_history.append(smallist_distance)
    nominal_total_maximum_distance_history.append(largiest_distance)
    nominal_feasibility_list.append(True)
    nominal_failure_reason_list.append('collision' if collision_time is not None else 'ok')
    nominal_failure_time_list.append(collision_time)
    nominal_initial_position_std_dev_list.append(initial_position_std_dev)
    nominal_initial_velocity_std_dev_list.append(initial_velocity_std_dev)
    nominal_final_position_std_dev_list.append(final_position_std_dev)
    nominal_final_velocity_std_dev_list.append(final_velocity_std_dev)

    print(f"Test case {test_case} completed.")
    print(f"\n--- Simulation Summary ---")
    print(f"Feasible cases: {test_case + 1} / {test_case_num}")
    if test_case_num > 0:
        print(f"Feasibility Rate: {(test_case + 1)/test_case_num*100:.2f}%")

In [ ]:

nominal_data = {
    'nominal_total_history': nominal_total_history,
    'nominal_total_control_history': nominal_total_control_history,
    'nominal_total_minimum_distance_history': nominal_total_minimum_distance_history,
    'nominal_total_maximum_distance_history': nominal_total_maximum_distance_history,
    'nominal_feasibility_list': nominal_feasibility_list,
    'nominal_initial_position_std_dev_list': nominal_initial_position_std_dev_list,
    'nominal_initial_velocity_std_dev_list': nominal_initial_velocity_std_dev_list,
    'nominal_final_position_std_dev_list': nominal_final_position_std_dev_list,
    'nominal_final_velocity_std_dev_list': nominal_final_velocity_std_dev_list,
    'nominal_failure_reason_list': nominal_failure_reason_list,
    'nominal_failure_time_list': nominal_failure_time_list
}

with open(f"result/{CASE_ID}/nominal/nominal_simulation_data.pkl", "wb") as f:
    pickle.dump(nominal_data, f)
    print("Saved! You should see nominal_simulation_data.pkl in the folder.")